In [1]:
from langchain_groq import ChatGroq
import yfinance as yf
import pandas as pd

In [ ]:
llm = ChatGroq(
    api_key="Replace With Yours",
    model="meta-llama/llama-4-scout-17b-16e-instruct"
)

print("LLM Ready!")

LLM Ready!


In [3]:
def get_stock_data(ticker, period="1mo"):
    stock = yf.Ticker(ticker)
    data = stock.history(period=period)
    data = data[["Open", "High", "Low", "Close", "Volume"]]
    data.index = data.index.tz_localize(None)
    return data

def get_stock_info(ticker):
    stock = yf.Ticker(ticker)
    info = stock.info
    return {
        "Company": info.get("longName", "N/A"),
        "Sector": info.get("sector", "N/A"),
        "Industry": info.get("industry", "N/A"),
        "Market Cap": info.get("marketCap", "N/A"),
        "PE Ratio": info.get("trailingPE", "N/A"),
        "52W High": info.get("fiftyTwoWeekHigh", "N/A"),
        "52W Low": info.get("fiftyTwoWeekLow", "N/A"),
        "Current Price": info.get("currentPrice", "N/A")
    }

def get_stock_news(ticker, num_articles=5):
    stock = yf.Ticker(ticker)
    news = stock.news
    articles = []
    for article in news[:num_articles]:
        articles.append({
            "title": article.get("content", {}).get("title", "N/A"),
            "summary": article.get("content", {}).get("summary", "N/A"),
        })
    return pd.DataFrame(articles)

print("All functions loaded!")

All functions loaded!


In [4]:
def analyze_stock(ticker):
    # Gather all data
    info = get_stock_info(ticker)
    df = get_stock_data(ticker, period="1mo")
    news_df = get_stock_news(ticker, num_articles=5)
    
    # Prepare price summary
    latest_price = df["Close"].iloc[-1]
    price_change = df["Close"].iloc[-1] - df["Close"].iloc[0]
    price_change_pct = (price_change / df["Close"].iloc[0]) * 100
    
    # Prepare news headlines
    headlines = "\n".join([f"- {row['title']}" for _, row in news_df.iterrows()])
    
    # Build prompt
    prompt = f"""
    You are an expert financial analyst. Analyze the following stock data and provide a clear investment insight.

    Company: {info['Company']}
    Sector: {info['Sector']}
    Current Price: ${info['Current Price']}
    PE Ratio: {info['PE Ratio']}
    Market Cap: {info['Market Cap']}
    52W High: {info['52W High']}
    52W Low: {info['52W Low']}
    
    Price Change (1 Month): ${price_change:.2f} ({price_change_pct:.2f}%)
    
    Latest News Headlines:
    {headlines}
    
    Please provide:
    1. Overall sentiment (Bullish/Bearish/Neutral)
    2. Key strengths
    3. Key risks
    4. Short term outlook
    5. Final recommendation
    """
    
    response = llm.invoke(prompt)
    return response.content

print("Analysis function ready!")

Analysis function ready!


In [5]:
analysis = analyze_stock("AAPL")
print(analysis)

**Analysis of Apple Inc. Stock**

### 1. Overall Sentiment
The overall sentiment for Apple Inc. stock is **Bullish**. This conclusion is based on the recent price increase, positive news headlines, and the company's strong market position.

### 2. Key Strengths
- **Strong Market Position**: Apple Inc. is a leader in the technology sector with a significant market capitalization of $3,983,727,394,816.
- **Recent Price Momentum**: The stock has shown a positive price change of $15.72 (6.15%) in the last month, indicating growing investor confidence.
- **Innovation and Product Pipeline**: News about the iPhone 17 and MacBook Neo driving forecast suggests that Apple is continuing to innovate and potentially expand its product lineup, which could be a growth driver.
- **Stable Financials**: A PE ratio of 34.35 suggests that while the stock may be considered somewhat pricey, it is in line with many tech stocks and reflects investor expectations for future growth.

### 3. Key Risks
- **High V

In [6]:
analysis = analyze_stock("TSLA")
print(analysis)

**Analysis of Tesla, Inc. Stock Data**

### 1. Overall Sentiment
The overall sentiment for Tesla, Inc. stock is **Neutral**. This assessment is based on the current price, valuation metrics, recent price movements, and the mixed signals from the latest news headlines.

### 2. Key Strengths
- **Market Leadership**: Tesla is a leader in the electric vehicle (EV) market, with a strong brand and innovative products.
- **Diversified Revenue Streams**: Beyond vehicle sales, Tesla generates revenue from energy solutions (solar panels, batteries) and services, including its autonomous driving technology.
- **Financial Performance**: The company has shown significant growth in sales, with notable contributions from its other ventures such as SpaceX and xAI, indicating a diversified and robust financial structure.
- **Market Capitalization**: With a market cap of $1.433 trillion, Tesla has significant market presence and liquidity.

### 3. Key Risks
- **High Valuation**: The PE ratio of 343.81 i